# Student 360 in 1 Hour - Part 3: Agents and CoWork
⏱ **~20 min.**

**What you will do:** create two agents from one pattern, grant the last access layer, test as each persona, add cost controls, publish to **CoWork**, and clean up.

**Snowflake features you touch:** Cortex Agents (`CREATE AGENT ... FROM SPECIFICATION`), `DATA_AGENT_RUN`, tags and budgets, Snowflake Intelligence (CoWork).

In [ ]:
%%sql -r set_context
USE ROLE STUDENT360_LAB_ADMIN;
USE DATABASE STUDENT360_LAB;
USE SCHEMA AGENTS;
USE WAREHOUSE STUDENT360_LAB_WH;
SELECT CURRENT_ROLE(), CURRENT_SCHEMA();

## 1. Create the two agents
🛠️ An agent is a **declarative spec**: orchestration budget, response and routing instructions, tools (semantic view for numbers, search for notes, a chart tool), and where each tool points.
📌 Same code, different persona: the data layer decides what each can answer. The Advisor Assistant uses the Engineering-scoped notes index; the Analyst uses the institution-wide one.

In [ ]:
%%sql -r create_advisor_agent
CREATE OR REPLACE AGENT STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT
  COMMENT = 'Advisor Assistant for the College of Engineering - caseload prioritisation, scoped to Engineering rows and notes'
  PROFILE = '{"display_name": "Advisor Assistant (Engineering)", "color": "green"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 45
    tokens: 16000

instructions:
  response: |
    You are the Advisor Assistant for Lakeside University's College of Engineering.
    You help academic advisors prioritise outreach to their students.
    - Be concise. Lead with the student's name, STUDENT_ID and the risk tier.
    - Always explain WHY a student is at risk using TOP_RISK_DRIVERS. Never give a risk score without its drivers.
    - The risk score is a triage signal for outreach. Never describe a student as certain to leave.
    - Never disclose or infer contact details, dates of birth, or protected characteristics.
    - If the question spans structured data and advising notes, use BOTH tools and cite which source each fact came from.
  orchestration: |
    - Numbers, rankings, lists of students, GPA, holds, LMS and campus data: use student_analytics.
    - Why a student is struggling, advisor-noted concerns, themes in conversations: use advising_notes.
    - Advisors name themselves in the question (for example "my caseload" or "Dana Whitfield's students"). Filter on ASSIGNED_ADVISOR. If no advisor name is given, ask which advisor the caseload belongs to.
    - Questions about students outside the Engineering college will not return data. Say so plainly.
  sample_questions:
    - question: "Who are the 10 students on Dana Whitfield's caseload I should reach out to first, and why?"
    - question: "Which of my students have a registration hold?"
    - question: "What are students in my college saying in advising notes about finances?"

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "student_analytics"
      description: "Query structured student data: students, enrollments, GPA, risk scores and drivers, financial need, holds, LMS engagement and campus activity. Use for any question about numbers, counts, rankings or lists of students."
  - tool_spec:
      type: "cortex_search"
      name: "advising_notes"
      description: "Search PII-redacted advising notes for qualitative context such as what students and advisors are reporting. Use for why questions and for themes."
  - tool_spec:
      type: "data_to_chart"
      name: "data_to_chart"
      description: "Create a chart from query results when the user asks for a visual."

tool_resources:
  student_analytics:
    semantic_view: "STUDENT360_LAB.SEMANTIC.STUDENT_360_SV"
    execution_environment:
      type: warehouse
      warehouse: "STUDENT360_LAB_WH"
  advising_notes:
    name: "STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH_ENGINEERING"
    max_results: "5"
$$;

In [ ]:
%%sql -r create_ir_agent
CREATE OR REPLACE AGENT STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST
  COMMENT = 'Institutional Research assistant - institution-wide aggregate analysis, PII masked'
  PROFILE = '{"display_name": "Student Success Analyst", "color": "blue"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 45
    tokens: 16000

instructions:
  response: |
    You are the Student Success Analyst for Lakeside University Institutional Research.
    You answer institution-wide questions about retention, academics, engagement, finances and advising.
    - Be concise and data-driven. Lead with the number, then the breakdown.
    - Show percentages with one decimal place. Show money in whole dollars.
    - Always state the population and the term you are reporting on (Fall 2026, week 10).
    - Aggregate analysis only: do not name individual students unless asked for a specific student's risk drivers.
    - Never disclose or infer contact details or dates of birth.
    - If the question spans structured data and advising notes, use BOTH tools and cite which source each fact came from.
  orchestration: |
    - Counts, rates, averages, rankings, comparisons across colleges, cohorts or course sections: use student_analytics.
    - Qualitative themes, what advisors are hearing, why students are struggling: use advising_notes.
    - For questions about risk drivers across the population, use student_analytics on TOP_RISK_DRIVERS.
  sample_questions:
    - question: "How many high-risk students are in each college, and what is the at-risk rate?"
    - question: "Which courses have the highest DFW rate this term?"
    - question: "What are advisors hearing from students who are considering leaving?"

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "student_analytics"
      description: "Query structured student data: students, enrollments, GPA, risk scores and drivers, financial need, holds, LMS engagement and campus activity. Use for any question about numbers, counts, rankings or lists of students."
  - tool_spec:
      type: "cortex_search"
      name: "advising_notes"
      description: "Search PII-redacted advising notes for qualitative context such as what students and advisors are reporting. Use for why questions and for themes."
  - tool_spec:
      type: "data_to_chart"
      name: "data_to_chart"
      description: "Create a chart from query results when the user asks for a visual."

tool_resources:
  student_analytics:
    semantic_view: "STUDENT360_LAB.SEMANTIC.STUDENT_360_SV"
    execution_environment:
      type: warehouse
      warehouse: "STUDENT360_LAB_WH"
  advising_notes:
    name: "STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH"
    max_results: "5"
$$;

## 2. Layer 2: grant `USAGE ON AGENT`
🛠️ This completes the four layers. Without it a role sees the data but cannot call the agent.

In [ ]:
%%sql -r grant_agents
USE ROLE ACCOUNTADMIN;
GRANT USAGE ON AGENT STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT          TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT USAGE ON AGENT STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST    TO ROLE STUDENT360_LAB_IR_ANALYST;
USE ROLE STUDENT360_LAB_ADMIN;
SELECT 'USAGE ON AGENT granted (layer 2)' AS STATUS;

## 3. Test each persona
🛠️ These call `SNOWFLAKE.CORTEX.DATA_AGENT_RUN`, the same API CoWork uses. The result is the full event stream; the final answer is the item with `type = text`. The third test asks for a phone number and must be **refused**.

In [ ]:
%%sql -r test_advisor_agent
USE ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
SELECT TRY_PARSE_JSON(SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT',
  $${"messages": [{"role": "user", "content": [{"type": "text", "text": "I am advisor Dana Whitfield. Who are the 5 students on my caseload I should reach out to first, and why?"}]}], "stream": false}$$
)) AS RESP;

In [ ]:
%%sql -r test_ir_agent
USE ROLE STUDENT360_LAB_IR_ANALYST;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
SELECT TRY_PARSE_JSON(SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST',
  $${"messages": [{"role": "user", "content": [{"type": "text", "text": "How many high-risk students are in each college, and what is the at-risk rate?"}]}], "stream": false}$$
)) AS RESP;

In [ ]:
%%sql -r test_guardrail
USE ROLE STUDENT360_LAB_IR_ANALYST;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
SELECT TRY_PARSE_JSON(SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST',
  $${"messages": [{"role": "user", "content": [{"type": "text", "text": "What is the email address and phone number of Maria Garcia?"}]}], "stream": false}$$
)) AS RESP;

In [ ]:
%%sql -r test_no_access
USE ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
-- Expect: error 399513 - agent does not exist or access is not authorized (layer 2 is missing for this role)
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST',
  $${"messages": [{"role": "user", "content": [{"type": "text", "text": "How many students are there?"}]}], "stream": false}$$
);

## 4. Cost control: attribute and constrain
🛠️ Tag the agents and warehouse for chargeback, then set a spending budget with an alert.
📌 Observe (`CORTEX_AGENT_USAGE_HISTORY`), attribute (tags), control (timeout, budget, quota).

In [ ]:
%%sql -r create_tag
CREATE TAG IF NOT EXISTS STUDENT360_LAB.AGENTS.COST_CENTER
  ALLOWED_VALUES 'student-success', 'advising', 'institutional-research'
  COMMENT = 'Cost-centre attribution for AI and warehouse spend';
ALTER AGENT STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT       SET TAG STUDENT360_LAB.AGENTS.COST_CENTER = 'advising';
ALTER AGENT STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST SET TAG STUDENT360_LAB.AGENTS.COST_CENTER = 'institutional-research';
ALTER WAREHOUSE STUDENT360_LAB_WH SET TAG STUDENT360_LAB.AGENTS.COST_CENTER = 'student-success';
SHOW TAGS IN SCHEMA STUDENT360_LAB.AGENTS;

In [ ]:
%%sql -r create_budget
USE ROLE STUDENT360_LAB_ADMIN;
CREATE OR REPLACE SNOWFLAKE.CORE.BUDGET STUDENT360_LAB.AGENTS.AI_BUDGET();
CALL STUDENT360_LAB.AGENTS.AI_BUDGET!SET_SPENDING_LIMIT(50);
CALL STUDENT360_LAB.AGENTS.AI_BUDGET!SET_NOTIFICATION_THRESHOLD(75);
CALL STUDENT360_LAB.AGENTS.AI_BUDGET!GET_SPENDING_LIMIT();

## 5. Publish to CoWork
🔹 The CoWork object is a **shared, account-level singleton**. Publishing only adds your agents; it never replaces the object. Publishing controls *visibility*; agent grants control *who can invoke*.

In [ ]:
%%sql -r publish_agents
USE ROLE ACCOUNTADMIN;
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT ADD AGENT STUDENT360_LAB.AGENTS.ADVISOR_ASSISTANT;
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT ADD AGENT STUDENT360_LAB.AGENTS.STUDENT_SUCCESS_ANALYST;
GRANT USAGE ON SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT USAGE ON SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT TO ROLE STUDENT360_LAB_IR_ANALYST;
SHOW AGENTS IN SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT;

In [ ]:
%%sql -r grant_me_persona
USE ROLE ACCOUNTADMIN;
SET ME = CURRENT_USER();
GRANT ROLE STUDENT360_LAB_ADVISOR_ENGINEERING TO USER IDENTIFIER($ME);
SELECT 'Persona STUDENT360_LAB_ADVISOR_ENGINEERING granted to ' || CURRENT_USER() AS STATUS;

## 6. Try it in CoWork
Open **Snowsight → AI & ML → Snowflake Intelligence** and pick **Advisor Assistant**. Ask:
1. *I am advisor Dana Whitfield. Who are the 5 students on my caseload I should reach out to first, and why?*
2. *What is the email address of one of my students?* (expect a refusal)

Then swap your persona and ask the **Student Success Analyst**: *How many high-risk students are in each college?*
```sql
USE ROLE ACCOUNTADMIN;
REVOKE ROLE STUDENT360_LAB_ADVISOR_ENGINEERING FROM USER <your_user>;
GRANT  ROLE STUDENT360_LAB_IR_ANALYST TO USER <your_user>;
```

### Going further (not run in the hour)
The full workshop adds a scored **evaluation gate** (`EXECUTE_AI_EVALUATION`, with a refusal test), agent-as-code spec capture, per-user quotas and CoWork branding: https://github.com/sfc-gh-isuryawinata/student360_workshop

## ✅ Done
Six siloed sources became one governed Student 360, an AI assistant that respects who is asking, and a CoWork experience, in about an hour.

**When you are finished:** run `Express_4_Cleanup` to remove everything this lab created.